# Graph-Augmented ERC: Final Experiment Pipeline

**Paper:** From Chunk Retrieval to Relation-Path Grounding: Graph-Augmented Foundation Models for Social Emotion Reasoning  
**Dataset:** MELD (primary), EmoryNLP (secondary)  
**Methods:** llm_only, full_context_llm, bm25_rag, dense_rag, text_only_rag, graph_rag  

---
### ⚠️ Before Running
1. Set `OPENAI_API_KEY` in `.env` file
2. Set `LLM_MODEL` and `JUDGE_MODEL` below (default: `gpt-4o` for final)
3. Set `MAX_SAMPLES` for faithfulness eval (default: 500)

### 📌 Key Changes vs mini experiment
- **Prompt updated**: evidence citation is now MANDATORY (improves faithfulness)
- **Model**: gpt-4o (final paper submission)
- **Parallel faithfulness eval**: 8 workers

## 0. Setup

In [ ]:
import sys, os, json
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.chdir(PROJECT_ROOT)

from dotenv import load_dotenv
load_dotenv()

# ── ✅ Model selection ───────────────────────────────────────────────────────
LLM_MODEL   = "gpt-4o"       # final paper: "gpt-4o" | dev: "gpt-4o-mini"
JUDGE_MODEL = "gpt-4o"       # final paper: "gpt-4o" | dev: "gpt-4o-mini"
DATASET     = "meld"
SPLIT       = "test"
MAX_SAMPLES = 500             # instances per method for faithfulness eval

from config import CFG
CFG.llm_model   = LLM_MODEL
CFG.judge_model = JUDGE_MODEL

print(f"{'LLM model':<15}: {CFG.llm_model}")
print(f"{'Judge model':<15}: {CFG.judge_model}")
print(f"{'Dataset':<15}: {DATASET} / {SPLIT}")
print(f"{'Faith samples':<15}: {MAX_SAMPLES} per method")
print()
print("📌 Prompt: evidence citation ENFORCED (updated from mini experiment)")

## 1. Load Data

In [ ]:
from data_loader import load_dataset, save_dialogues, load_dialogues
from preprocessing import build_instances, filter_valid

processed_path = Path(CFG.data_dir) / "processed" / DATASET / f"{SPLIT}_dialogues.json"

if processed_path.exists():
    dialogues = load_dialogues(processed_path)
    print(f"Loaded {len(dialogues)} dialogues from cache")
else:
    dialogues = load_dataset(DATASET, SPLIT)
    save_dialogues(dialogues, processed_path)
    print(f"Loaded {len(dialogues)} dialogues from raw data")

instances = build_instances(dialogues, DATASET)
instances = filter_valid(instances, DATASET)
print(f"Instances: {len(instances)}")

## 2. Load Graphs

In [ ]:
from graph_builder import build_all_graphs, save_graphs, load_graphs

graph_path = Path(CFG.data_dir) / "graphs" / DATASET / f"{SPLIT}_graphs.pkl"

if graph_path.exists():
    graphs = load_graphs(graph_path)
    print(f"Loaded {len(graphs)} graphs from cache")
else:
    print("Building graphs...")
    graphs = build_all_graphs(dialogues, instances, audio_cache=None)
    save_graphs(graphs, graph_path)
    print(f"Built and saved {len(graphs)} graphs")

## 3. Run Inference

> Skips methods with existing output files. **Delete a file to re-run it.**  
> To run one method: change `methods_to_run = ["graph_rag"]`

In [ ]:
from llm_runner import run_experiment, load_predictions, METHODS

outdir = Path(CFG.output_dir) / "predictions" / DATASET / SPLIT
outdir.mkdir(parents=True, exist_ok=True)

methods_to_run = METHODS  # or set to ["graph_rag"] for single method

for method in methods_to_run:
    out_path = outdir / f"{method}.json"
    if out_path.exists():
        n = len(load_predictions(out_path))
        print(f"  [SKIP] {method:<20}: {n} predictions already saved")
        continue
    print(f"  [RUN ] {method}...")
    run_experiment(instances, method, graphs, out_path)
    print(f"  [DONE] {method}")

## 4. Emotion Prediction Metrics

In [ ]:
import pandas as pd
from evaluator import compute_metrics, compare_methods, save_metrics
from utils import save_json

results_by_method = {}
for method in METHODS:
    path = outdir / f"{method}.json"
    if path.exists():
        results_by_method[method] = load_predictions(path)

metrics_dir = Path(CFG.output_dir) / "tables" / DATASET / SPLIT
metrics_dir.mkdir(parents=True, exist_ok=True)

all_metrics = {}
for method, preds in results_by_method.items():
    m = compute_metrics(preds, DATASET)
    all_metrics[method] = m
    save_metrics(m, metrics_dir / f"{method}_metrics.json")

rows = []
for method, m in all_metrics.items():
    rows.append({
        "Method":      method,
        "Accuracy":    m.get("accuracy"),
        "Macro F1":    m.get("macro_f1"),
        "Weighted F1": m.get("weighted_f1"),
    })

df = pd.DataFrame(rows).set_index("Method").round(4)
print("=== Emotion Prediction Results ===")
print(df.to_string())

save_json(compare_methods(results_by_method, DATASET), metrics_dir / "comparison.json")

## 5. Faithfulness Evaluation (Claim-Level)

**Updated prompt enforces evidence citation** → expect higher faithfulness scores vs mini experiment.

Pipeline:
1. Decompose explanation → atomic claims  
2. Verify each claim against retrieved evidence → `supported`=1.0, `partially`=0.5, `unsupported`=0.0  
3. `faithfulness_score` = mean claim score  
4. Parallel (8 workers) for speed

In [ ]:
from grounding_evaluator import evaluate_faithfulness, summarize_faithfulness

grounding_dir = Path(CFG.output_dir) / "grounding" / DATASET / SPLIT
grounding_dir.mkdir(parents=True, exist_ok=True)

faithfulness_summaries = {}

for method, preds in results_by_method.items():
    judged_path = grounding_dir / f"{method}_judged.json"
    if judged_path.exists():
        with open(judged_path) as f:
            judged = json.load(f)
        print(f"  [SKIP] {method}: {len(judged)} already judged")
    else:
        sample = preds[:MAX_SAMPLES]
        print(f"  [RUN ] {method} ({len(sample)} predictions)...")
        judged = evaluate_faithfulness(sample, instances, out_path=judged_path, max_workers=8)

    summary = summarize_faithfulness(judged)
    faithfulness_summaries[method] = summary
    save_json(summary, grounding_dir / f"{method}_summary.json")

print("\nDone.")

## 6. Results Table

In [ ]:
rows = []
for method in METHODS:
    if method not in all_metrics:
        continue
    m  = all_metrics[method]
    fs = faithfulness_summaries.get(method, {})
    rows.append({
        "Method":       method,
        "Macro F1":     m.get("macro_f1"),
        "Weighted F1":  m.get("weighted_f1"),
        "Accuracy":     m.get("accuracy"),
        "Faithfulness": (fs.get("faithfulness_score") or {}).get("mean"),
        "Supported%":   (fs.get("supported_ratio")    or {}).get("mean"),
        "Unsupported%": (fs.get("unsupported_ratio")  or {}).get("mean"),
    })

df_final = pd.DataFrame(rows).set_index("Method").round(4)
print(f"=== FINAL RESULTS — {DATASET.upper()} / {LLM_MODEL} ===")
print(df_final.to_string())

save_json(
    df_final.reset_index().to_dict(orient="records"),
    metrics_dir / f"final_summary_{LLM_MODEL.replace('-','_')}.json"
)
print(f"\nSaved → {metrics_dir}")

## 7. Visualization

In [ ]:
import matplotlib.pyplot as plt

methods      = list(all_metrics.keys())
macro_f1     = [all_metrics[m].get("macro_f1", 0) for m in methods]
faith_scores = [
    (faithfulness_summaries.get(m, {}).get("faithfulness_score") or {}).get("mean", 0)
    for m in methods
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(methods, macro_f1, color="steelblue")
axes[0].set_title(f"Emotion Prediction — Macro F1 ({LLM_MODEL})", fontsize=12)
axes[0].set_ylabel("Macro F1")
axes[0].set_ylim(0, 0.8)
axes[0].tick_params(axis="x", rotation=30)
for i, v in enumerate(macro_f1):
    axes[0].text(i, v + 0.005, f"{v:.3f}", ha="center", fontsize=9)

axes[1].bar(methods, faith_scores, color="coral")
axes[1].set_title(f"Faithfulness Score — Claim-Level ({LLM_MODEL})", fontsize=12)
axes[1].set_ylabel("Mean Faithfulness Score")
axes[1].set_ylim(0, 1)
axes[1].tick_params(axis="x", rotation=30)
for i, v in enumerate(faith_scores):
    axes[1].text(i, v + 0.005, f"{v:.3f}", ha="center", fontsize=9)

plt.tight_layout()
fig_path = Path(CFG.output_dir) / "figures" / f"{DATASET}_{SPLIT}_{LLM_MODEL}_results.png"
fig_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {fig_path}")

## 8. Ablation Study

In [ ]:
# Run ablation variants first:
#   python run_ablation.py --dataset meld --split test --max-instances 500

ablation_variants = [
    ("graph_rag",                  "graph_rag (full)"),
    ("graph_rag_no_same_speaker",  "- w/o same_speaker"),
    ("graph_rag_no_emotion_shift", "- w/o emotion_shift"),
    ("graph_rag_no_audio",         "- w/o audio"),
    ("text_only_rag",              "text_only_rag (no graph)"),
]

abl_rows = []
base_f1  = None
for fname, label in ablation_variants:
    mp = metrics_dir   / f"{fname}_metrics.json"
    sp = grounding_dir / f"{fname}_summary.json"
    if not mp.exists():
        print(f"  [MISSING] {fname}_metrics.json — run run_ablation.py first")
        continue
    m = json.load(open(mp))
    s = json.load(open(sp)) if sp.exists() else {}
    mf1   = m.get("macro_f1", 0)
    faith = (s.get("faithfulness_score") or {}).get("mean", 0) or 0
    if base_f1 is None:
        base_f1 = mf1
    abl_rows.append({
        "Variant":  label,
        "Faith":    round(faith, 4),
        "Macro F1": round(mf1, 4),
        "Accuracy": round(m.get("accuracy", 0), 4),
        "Δ MacF1":  round(mf1 - base_f1, 4),
    })

if abl_rows:
    df_abl = pd.DataFrame(abl_rows).set_index("Variant")
    print("=== Ablation Study ===")
    print(df_abl.to_string())

## 9. Judge Reliability (Inter-Model Agreement)

Validates LLM-as-judge by computing Spearman ρ between GPT-4o and GPT-4o-mini judges.  
Run on **50-sample** subset to save cost.

In [ ]:
from grounding_evaluator import validate_judge_reliability

reliability_path = grounding_dir / "judge_reliability.json"

if reliability_path.exists():
    reliability = json.load(open(reliability_path))
    print("[SKIP] Already computed:")
else:
    sample_preds = results_by_method.get("graph_rag", [])[:50]
    if sample_preds:
        print("Running inter-model reliability (GPT-4o vs GPT-4o-mini, n=50)...")
        reliability = validate_judge_reliability(
            sample_preds, instances,
            judge_a="gpt-4o",
            judge_b="gpt-4o-mini",
        )
        save_json(reliability, reliability_path)
    else:
        print("graph_rag predictions not available.")
        reliability = {}

print("Spearman ρ (faithfulness_score):", reliability)

## 10. EmoryNLP (Secondary Dataset — Optional)

In [ ]:
# Uncomment to run EmoryNLP experiments

# DATASET_2 = "emorynlp"
# outdir_2  = Path(CFG.output_dir) / "predictions" / DATASET_2 / "test"
# outdir_2.mkdir(parents=True, exist_ok=True)
# processed_path_2 = Path(CFG.data_dir) / "processed" / DATASET_2 / "test_dialogues.json"
# dialogues_2 = load_dialogues(processed_path_2)
# instances_2 = filter_valid(build_instances(dialogues_2, DATASET_2), DATASET_2)
# graph_path_2 = Path(CFG.data_dir) / "graphs" / DATASET_2 / "test_graphs.pkl"
# graphs_2 = load_graphs(graph_path_2) if graph_path_2.exists() else build_all_graphs(dialogues_2, instances_2)
# for method in METHODS:
#     out_path = outdir_2 / f"{method}.json"
#     if not out_path.exists():
#         run_experiment(instances_2, method, graphs_2, out_path)

print("EmoryNLP section ready — uncomment to activate.")

## 11. Mini vs Final Comparison

Compare gpt-4o-mini (dev) vs gpt-4o (final) results side by side.

In [ ]:
mini_summary  = metrics_dir / "final_summary_gpt_4o_mini.json"
final_summary = metrics_dir / f"final_summary_{LLM_MODEL.replace('-','_')}.json"

if mini_summary.exists() and final_summary.exists():
    mini  = {r["Method"]: r for r in json.load(open(mini_summary))}
    final = {r["Method"]: r for r in json.load(open(final_summary))}
    rows = []
    for method in METHODS:
        if method not in mini or method not in final:
            continue
        rows.append({
            "Method":          method,
            "mini MacF1":      mini[method].get("Mac-F1"),
            "4o MacF1":        final[method].get("Mac-F1"),
            "Δ MacF1":         round((final[method].get("Mac-F1") or 0) - (mini[method].get("Mac-F1") or 0), 4),
            "mini Faith":      mini[method].get("Faith"),
            "4o Faith":        final[method].get("Faith"),
            "Δ Faith":         round((final[method].get("Faith") or 0) - (mini[method].get("Faith") or 0), 4),
        })
    df_cmp = pd.DataFrame(rows).set_index("Method").round(4)
    print("=== gpt-4o-mini vs gpt-4o ===")
    print(df_cmp.to_string())
else:
    print("Run both mini and final experiments to see comparison.")